# **1. Perkenalan Dataset**

## Customer Churn Prediction

Proyek ini bertujuan untuk memprediksi **churn pelanggan** (pelanggan yang berhenti berlangganan) menggunakan dataset **Customer Churn, Uplift & Feedback Dataset** dari Kaggle.

### Sumber Dataset
Dataset diperoleh dari [Kaggle - Customer Churn, Uplift and Feedback Dataset](https://www.kaggle.com/datasets/harrachimustapha/customer-churn-uplift-and-feedback-dataset) yang dikumpulkan oleh **harrachimustapha**.

### Deskripsi Dataset
Dataset ini berisi data pelanggan telekomunikasi yang mencakup berbagai informasi seperti:
- **Informasi Demografis**: state, area_code, account_length
- **Informasi Layanan**: international_plan, voice_mail_plan, jumlah pesan voicemail
- **Pola Penggunaan**: total menit, panggilan, dan tagihan per periode (day/eve/night/intl)
- **Informasi Akun**: jumlah panggilan ke layanan pelanggan
- **Fitur Turunan**: usage_intensity, customer_value_segment, rule_based_churn_risk_score
- **Target**: `churn` (1 = churn/berhenti, 0 = tidak churn)

### Struktur Data
Dataset terbagi menjadi:
- **train.csv**: 2.666 baris, 34 kolom — data pelatihan
- **test.csv**: 667 baris, 34 kolom — data pengujian
- **full_customers.csv**: 3.333 baris — gabungan train + test

Dataset tambahan:
- **customer_feedback.csv**: Feedback teks, kategori, sentimen, intensitas komplain
- **campaign_uplift.csv**: Data kampanye retensi (treatment_group, offer_type, uplift_label)
- **business_costs.csv**: Data biaya bisnis (monthly_revenue, CLV, retention_cost)

### Tujuan
Membangun model **Churn Prediction** untuk mengidentifikasi pelanggan yang berpotensi churn sehingga perusahaan dapat mengambil tindakan pencegahan.

# **2. Import Library**

Mengimpor pustaka Python yang dibutuhkan untuk analisis data, visualisasi, dan preprocessing.

In [ ]:
# Import library dasar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import library preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

# Import library evaluasi
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve
)

# Import model (akan digunakan setelah preprocessing)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Setting visualisasi
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Semua library berhasil diimpor!')


# **3. Memuat Dataset**

Memuat dataset dari file CSV ke dalam DataFrame pandas.

In [ ]:
# ============================================
# LOAD DATASET - Download raw data ke folder ../customer_churn_raw/
# ============================================

import os

# Setup path relatif terhadap lokasi notebook di folder preprocessing/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_DIR = os.path.join(NOTEBOOK_DIR, '..')
RAW_DATA_DIR = os.path.join(PROJECT_DIR, 'customer_churn_raw')
PREPROCESSING_DIR = os.path.join(NOTEBOOK_DIR, 'customer_churn_preprocessing')

# Buat direktori jika belum ada
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PREPROCESSING_DIR, exist_ok=True)

# Download dataset menggunakan kagglehub
try:
    import kagglehub
    download_path = kagglehub.dataset_download('harrachimustapha/customer-churn-uplift-and-feedback-dataset')
    print(f'Dataset di-download ke: {download_path}')
    
    # Cari file CSV di path download
    source_path = download_path
    if not os.path.exists(os.path.join(download_path, 'train.csv')):
        items = os.listdir(download_path)
        for item in items:
            sub = os.path.join(download_path, item)
            if os.path.isdir(sub) and 'train.csv' in os.listdir(sub):
                source_path = sub
                print(f'Menggunakan subdirektori: {source_path}')
                break
    
    # Copy raw data ke folder customer_churn_raw/
    import shutil
    for item in os.listdir(source_path):
        src = os.path.join(source_path, item)
        dst = os.path.join(RAW_DATA_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print(f'Raw data disalin ke: {RAW_DATA_DIR}')
    
except Exception as e:
    print(f'KaggleHub error: {e}')
    # Fallback: gunakan data yang sudah ada di customer_churn_raw/
    print('Menggunakan data yang sudah ada di customer_churn_raw/')

# Load data dari folder customer_churn_raw/
train_df = pd.read_csv(f'{RAW_DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{RAW_DATA_DIR}/test.csv')
full_df = pd.read_csv(f'{RAW_DATA_DIR}/full_customers.csv')

# Load data tambahan (untuk eksplorasi lebih lanjut)
feedback_df = pd.read_csv(f'{RAW_DATA_DIR}/customer_feedback.csv')
campaign_df = pd.read_csv(f'{RAW_DATA_DIR}/campaign_uplift.csv')
costs_df = pd.read_csv(f'{RAW_DATA_DIR}/business_costs.csv')

print('\nDataset berhasil dimuat!')
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Full shape: {full_df.shape}')
print(f'Feedback shape: {feedback_df.shape}')
print(f'Campaign shape: {campaign_df.shape}')
print(f'Costs shape: {costs_df.shape}')
print(f'\nRaw data path: {RAW_DATA_DIR}')
print(f'Preprocessing output path: {PREPROCESSING_DIR}')


In [ ]:
# Tampilkan beberapa baris pertama dataset
print('=== 5 Baris Pertama Train Data ===')
display(train_df.head())

print('=== Informasi Dataset ===')
train_df.info()

print('\n=== Jumlah Missing Values ===')
missing = train_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Tidak ada missing values')

print('\n=== Nama Kolom ===')
for i, col in enumerate(train_df.columns, 1):
    print(f'{i:2d}. {col}')


# **4. Exploratory Data Analysis (EDA)**

Melakukan eksplorasi data untuk memahami karakteristik dataset sebelum preprocessing dan pemodelan.

## 4.1 Distribusi Target (Churn)

In [ ]:
# Analisis distribusi target churn
print('=== Distribusi Target Churn ===')
churn_counts = train_df['churn'].value_counts()
churn_pct = train_df['churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    'Jumlah': churn_counts,
    'Persentase': churn_pct
})
print(churn_summary)

# Visualisasi distribusi churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4CAF50', '#F44336']
bars = axes[0].bar(['Tidak Churn (0)', 'Churn (1)'], churn_counts.values,
                   color=colors, edgecolor='black')
for bar, count in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{count}\n({count/len(train_df)*100:.1f}%)',
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Distribusi Target Churn', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Pelanggan')

axes[1].pie(churn_counts.values, labels=['Tidak Churn', 'Churn'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0.02, 0.08), textprops={'fontsize': 12})
axes[1].set_title('Proporsi Churn', fontsize=14, fontweight='bold')

plt.suptitle('Analisis Class Imbalance', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Kesimpulan:')
print(f'- Data tidak seimbang (imbalanced): {churn_pct[0]:.1f}% tidak churn vs {churn_pct[1]:.1f}% churn')
print('- Perlu penanganan class imbalance saat modeling (SMOTE, class_weight, atau resampling)')


## 4.2 Analisis Statistik Deskriptif

In [ ]:
# Statistik deskriptif fitur numerik
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if 'churn' in num_cols:
    num_cols.remove('churn')

print('=== Statistik Deskriptif Fitur Numerik ===')
desc = train_df[num_cols].describe().T
desc['range'] = desc['max'] - desc['min']
desc['cv'] = (desc['std'] / desc['mean']).round(4)
print(desc.round(4))


In [ ]:
# Analisis statistik fitur kategorikal
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print('=== Fitur Kategorikal ===')
for col in cat_cols:
    if col == 'customer_id':
        continue
    print(f'\n{col}:')
    print(train_df[col].value_counts())


## 4.3 Analisis Hubungan Fitur dengan Churn

In [ ]:
# Korelasi fitur numerik dengan churn
print('=== Korelasi dengan Churn (Top 15) ===')
corr = train_df[num_cols].corrwith(train_df['churn']).abs().sort_values(ascending=False)
top15 = corr.head(15)
print(top15)

# Visualisasi korelasi
fig, ax = plt.subplots(figsize=(10, 6))
colors_corr = ['#F44336' if v > 0 else '#4CAF50' for v in corr.head(15).values]
bars = ax.barh(top15.index, top15.values, color=colors_corr)
ax.set_title('Top 15 Fitur - Korelasi Absolut dengan Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Korelasi Absolut')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap korelasi fitur numerik utama
key_features = [
    'account_length', 'total_day_minutes', 'total_day_charge',
    'total_eve_minutes', 'total_eve_charge',
    'total_night_minutes', 'total_night_charge',
    'total_intl_minutes', 'total_intl_charge',
    'customer_service_calls', 'number_vmail_messages',
    'total_minutes', 'total_charges', 'churn'
]

plt.figure(figsize=(12, 10))
corr_matrix = train_df[key_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Perbandingan distribusi churn vs non-churn (Boxplot)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

boxplot_features = [
    'total_day_minutes', 'total_eve_minutes', 'total_night_minutes',
    'total_intl_minutes', 'customer_service_calls', 'number_vmail_messages',
    'total_minutes', 'total_charges'
]

for i, col in enumerate(boxplot_features):
    ax = axes[i]
    df_melted = train_df[[col, 'churn']].copy()
    df_melted['churn'] = df_melted['churn'].map({0: 'Tidak Churn', 1: 'Churn'})
    sns.boxplot(data=df_melted, x='churn', y=col, ax=ax, palette=['#4CAF50', '#F44336'])
    ax.set_title(f'{col} vs Churn', fontsize=11, fontweight='bold')

plt.suptitle('Perbandingan Distribusi Fitur Numerik: Churn vs Non-Churn',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 4.4 Analisis Fitur Kategorikal terhadap Churn

In [ ]:
# Analisis churn rate berdasarkan fitur kategorikal
cat_analysis = ['international_plan', 'voice_mail_plan', 'area_code']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_analysis):
    ax = axes[i]
    ct = pd.crosstab(train_df[col], train_df['churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'], legend=True, edgecolor='black')
    ax.set_title(f'Churn Rate berdasarkan {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Persentase')
    ax.legend(['Tidak Churn', 'Churn'], loc='upper right')
    ax.tick_params(axis='x', rotation=45)
    for container in ax.containers:
        for bar in container:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
                        f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Pengaruh Fitur Kategorikal terhadap Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Analisis fitur turunan (engineered features)
engineered_cats = ['usage_intensity', 'customer_value_segment', 'rule_based_churn_risk_level', 'high_service_calls']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(engineered_cats):
    ax = axes[i]
    ct = pd.crosstab(train_df[col], train_df['churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'], legend=True, edgecolor='black')
    ax.set_title(f'Churn Rate berdasarkan {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Persentase')
    ax.legend(['Tidak Churn', 'Churn'], loc='upper right')
    ax.tick_params(axis='x', rotation=45)
    for container in ax.containers:
        for bar in container:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
                        f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Analisis Fitur Turunan terhadap Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 4.5 Analisis Data Tambahan (Feedback & Campaign)

In [ ]:
# Analisis feedback
print('=== Analisis Feedback ===')
print('\nKategori Feedback:')
print(feedback_df['feedback_category'].value_counts())
print('\nSentimen:')
print(feedback_df['sentiment'].value_counts())
print('\nStatistik Complaint Intensity:')
print(feedback_df['complaint_intensity'].describe())

# Visualisasi feedback
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

feedback_df['feedback_category'].value_counts().plot(
    kind='bar', ax=axes[0], color='#2196F3', edgecolor='black')
axes[0].set_title('Distribusi Kategori Feedback', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

feedback_df['sentiment'].value_counts().plot(
    kind='bar', ax=axes[1], color=['#4CAF50', '#FF9800', '#F44336'], edgecolor='black')
axes[1].set_title('Distribusi Sentimen', fontsize=12, fontweight='bold')

axes[2].hist(feedback_df['complaint_intensity'], bins=5,
             color='#9C27B0', edgecolor='black', alpha=0.7)
axes[2].set_title('Distribusi Complaint Intensity', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Intensitas')
axes[2].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()


In [ ]:
# Analisis campaign uplift
print('=== Analisis Campaign Uplift ===')
print('\nTreatment Group:')
print(campaign_df['treatment_group'].value_counts())
print('\nOffer Type:')
print(campaign_df['offer_type'].value_counts())
print('\nUplift Label:')
print(campaign_df['uplift_label'].value_counts())

# Visualisasi campaign
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

campaign_df['campaign_channel'].value_counts().plot(
    kind='bar', ax=axes[0], color='#FF5722', edgecolor='black')
axes[0].set_title('Distribusi Channel Kampanye', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

campaign_df['uplift_label'].value_counts().plot(
    kind='bar', ax=axes[1], color='#3F51B5', edgecolor='black')
axes[1].set_title('Distribusi Uplift Label', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# Contacted vs Responded
contact_respond = pd.crosstab(campaign_df['contacted'], campaign_df['responded'])
contact_respond.index = ['Not Contacted', 'Contacted']
contact_respond.columns = ['Not Responded', 'Responded']
contact_respond.plot(kind='bar', ax=axes[2], color=['#FF9800', '#4CAF50'], edgecolor='black')
axes[2].set_title('Contacted vs Responded', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Analisis biaya bisnis
print('=== Analisis Biaya Bisnis ===')
print(costs_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(costs_df['monthly_revenue'], bins=30,
             color='#2196F3', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribusi Monthly Revenue', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Monthly Revenue')

axes[1].hist(costs_df['estimated_clv'], bins=30,
             color='#FF9800', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Estimated CLV', fontsize=12, fontweight='bold')
axes[1].set_xlabel('CLV')

axes[2].hist(costs_df['retention_cost'], bins=30,
             color='#9C27B0', edgecolor='black', alpha=0.7)
axes[2].set_title('Distribusi Retention Cost', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Retention Cost')

plt.tight_layout()
plt.show()


# **5. Data Preprocessing**

Tahap preprocessing untuk membersihkan dan mempersiapkan data sebelum digunakan dalam model machine learning.

## 5.1 Pemisahan Fitur dan Target

In [ ]:
# ============================================
# PREPROCESSING PIPELINE
# ============================================

# Gunakan train_df sebagai data utama
df = train_df.copy()

# 1. Pisahkan fitur dan target
target = 'churn'
X = df.drop(columns=[target])
y = df[target]

print('Fitur (X):', X.shape)
print('Target (y):', y.shape)
print('\nDistribusi target setelah split:')
print(y.value_counts(normalize=True))


## 5.2 Identifikasi dan Penghapusan Kolom yang Tidak Diperlukan

In [ ]:
# 2. Hapus kolom yang tidak diperlukan untuk modeling
#    - customer_id: identifier unik, tidak informatif
#    - split: hanya berisi 1 nilai (train), tidak informatif
#    - Kolom duplikat: international_plan == has_international_plan, voice_mail_plan == has_voice_mail_plan
#    - Kolom turunan yang highly correlated dengan fitur asli (total_minutes, total_calls, total_charges)

cols_to_drop = [
    'customer_id',           # identifier unik
    'split',                 # hanya 1 nilai
    'has_international_plan', # duplikat dari international_plan
    'has_voice_mail_plan',    # duplikat dari voice_mail_plan
    'total_minutes',          # agregasi dari day+eve+night+intl minutes
    'total_calls',            # agregasi dari day+eve+night+intl calls
    'total_charges',          # agregasi dari day+eve+night+intl charges
    'avg_charge_per_minute',  # turunan
    'support_call_rate',      # turunan
]

X_clean = X.drop(columns=cols_to_drop)
print(f'Kolom dihapus ({len(cols_to_drop)}): {cols_to_drop}')
print(f'\nSisa kolom: {X_clean.shape[1]}')
print(f'Kolom: {X_clean.columns.tolist()}')


## 5.3 Deteksi dan Penanganan Outlier

In [ ]:
# 3. Deteksi outlier menggunakan IQR method
numeric_cols = X_clean.select_dtypes(include=[np.number]).columns.tolist()

print('=== Deteksi Outlier (IQR Method) ===')
outlier_info = []
for col in numeric_cols:
    Q1 = X_clean[col].quantile(0.25)
    Q3 = X_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((X_clean[col] < lower) | (X_clean[col] > upper)).sum()
    pct = outliers / len(X_clean) * 100
    if outliers > 0:
        outlier_info.append({'Fitur': col, 'Jumlah Outlier': outliers, 'Persentase': f'{pct:.2f}%'})

outlier_df = pd.DataFrame(outlier_info).sort_values('Jumlah Outlier', ascending=False)
print(outlier_df.to_string(index=False))
print(f'\nTotal fitur dengan outlier: {len(outlier_df)} dari {len(numeric_cols)} fitur numerik')


In [ ]:
# Visualisasi outlier dengan boxplot
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

outlier_cols = [
    'account_length', 'total_day_minutes', 'total_day_calls',
    'customer_service_calls', 'number_vmail_messages', 'total_intl_calls',
    'total_day_charge', 'total_eve_minutes'
]

for i, col in enumerate(outlier_cols):
    sns.boxplot(data=X_clean, y=col, ax=axes[i], color='#2196F3')
    axes[i].set_title(f'Boxplot - {col}', fontsize=10, fontweight='bold')

plt.suptitle('Deteksi Outlier dengan Boxplot', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Keputusan: Outlier tidak dihapus karena:')
print('1. Jumlah outlier relatif kecil (<5%) untuk sebagian besar fitur')
print('2. Beberapa outlier mungkin mencerminkan pola pelanggan yang valid (misal: heavy users)')
print('3. Model tree-based (Random Forest, Gradient Boosting) robust terhadap outlier')
print('4. Untuk model yang sensitif (Logistic Regression, SVM), scaling akan membantu mengurangi efek outlier')


## 5.4 Encoding Fitur Kategorikal

In [ ]:
# 4. Encoding fitur kategorikal
categorical_cols = X_clean.select_dtypes(include=['object']).columns.tolist()
print(f'Fitur kategorikal: {categorical_cols}')

# Label Encoding untuk binary categorical (Yes/No -> 1/0)
binary_cols = ['international_plan', 'voice_mail_plan']
for col in binary_cols:
    X_clean[col] = X_clean[col].map({'Yes': 1, 'No': 0})
    print(f'{col}: Yes=1, No=0')

# One-Hot Encoding untuk multi-category (state, usage_intensity, customer_value_segment, rule_based_churn_risk_level)
multi_cat_cols = [
    'state',
    'usage_intensity',
    'customer_value_segment',
    'rule_based_churn_risk_level'
]

# area_code juga bisa dianggap kategorikal
X_clean['area_code'] = X_clean['area_code'].astype(str)
multi_cat_cols.append('area_code')

# Lakukan One-Hot Encoding
X_encoded = pd.get_dummies(X_clean, columns=multi_cat_cols, drop_first=False, dtype=int)

print(f'\nSetelah encoding:')
print(f'Jumlah fitur: {X_encoded.shape[1]}')
print(f'\nTipe data setelah encoding:')
print(X_encoded.dtypes.value_counts())


## 5.5 Handling Missing Values

In [ ]:
# 5. Cek missing values
print('=== Cek Missing Values ===')
missing = X_encoded.isnull().sum()
if missing.sum() == 0:
    print('Tidak ada missing values. Data siap untuk langkah selanjutnya.')
else:
    missing_cols = missing[missing > 0]
    print(f'Ditemukan {len(missing_cols)} kolom dengan missing values:')
    print(missing_cols)
    # Imputasi dengan median untuk numerik, modus untuk kategorikal
    for col in missing_cols.index:
        if X_encoded[col].dtype in ['int64', 'float64']:
            X_encoded[col] = X_encoded[col].fillna(X_encoded[col].median())
            print(f'  - {col}: diimputasi dengan median')
        else:
            X_encoded[col] = X_encoded[col].fillna(X_encoded[col].mode()[0])
            print(f'  - {col}: diimputasi dengan modus')
    print('Missing values berhasil ditangani.')


## 5.6 Feature Scaling

In [ ]:
# 6. Feature Scaling
# StandardScaler untuk model yang sensitif terhadap skala (Logistic Regression, SVM, KNN)
# Tree-based model tidak memerlukan scaling, tapi kita siapkan saja untuk fleksibilitas

# Identifikasi kolom numerik yang perlu di-scale
scale_cols = [
    'account_length', 'area_code',
    'number_vmail_messages',
    'total_day_minutes', 'total_day_calls', 'total_day_charge',
    'total_eve_minutes', 'total_eve_calls', 'total_eve_charge',
    'total_night_minutes', 'total_night_calls', 'total_night_charge',
    'total_intl_minutes', 'total_intl_calls', 'total_intl_charge',
    'customer_service_calls', 'high_service_calls',
    'international_plan', 'voice_mail_plan',
    'rule_based_churn_risk_score'
]
# Hanya scale kolom yang benar-benar ada
scale_cols = [c for c in scale_cols if c in X_encoded.columns]

scaler = StandardScaler()
X_encoded[scale_cols] = scaler.fit_transform(X_encoded[scale_cols])

print(f'Feature Scaling selesai. {len(scale_cols)} kolom di-StandardScaler.')
print(f'\nContoh hasil scaling (5 baris pertama):')
print(X_encoded[scale_cols[:5]].head())


## 5.7 Train-Test Split

In [ ]:
# 7. Split data menjadi train dan validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Hasil Split ===')
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_val shape: {y_val.shape}')
print(f'\nDistribusi target train:')
print(y_train.value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print(f'\nDistribusi target validation:')
print(y_val.value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print(f'\nSplit selesai. Proporsi terjaga (stratified split).')


## 5.8 Eksperimen Baseline Model (Pra-Pemodelan)

In [ ]:
# 8. Baseline model comparison sebelum hyperparameter tuning
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(random_state=42, class_weight='balanced', probability=True)
}

results = []
print('=' * 80)
print(f'{"Baseline Model Comparison":^80}')
print('=' * 80)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_proba) if y_proba is not None else 0
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc:.4f}',
        'Precision': f'{prec:.4f}',
        'Recall': f'{rec:.4f}',
        'F1-Score': f'{f1:.4f}',
        'ROC-AUC': f'{auc:.4f}'
    })
    print(f'{name:25s} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

print('=' * 80)

results_df = pd.DataFrame(results)
print('\n=== Summary Hasil Baseline Model ===')
display(results_df)

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics_df = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    metrics_df[col] = metrics_df[col].astype(float)

metrics_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=axes[0], edgecolor='black', width=0.8)
axes[0].set_title('Perbandingan Metrik Model (Accuracy, Precision, Recall, F1)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Skor')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=30)

metrics_df.set_index('Model')[['ROC-AUC']].plot(
    kind='bar', ax=axes[1], color='#9C27B0', edgecolor='black', width=0.5, legend=False)
axes[1].set_title('Perbandingan ROC-AUC', fontsize=13, fontweight='bold')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_ylim(0.5, 1)
axes[1].tick_params(axis='x', rotation=30)
for container in axes[1].containers:
    for bar in container:
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, height + 0.01,
                     f'{height:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Baseline Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKesimpulan Baseline:')
print('- Model dengan performa terbaik berdasarkan F1-Score dan ROC-AUC akan dipilih untuk tuning lanjutan')
print('- Gradient Boosting dan Random Forest umumnya unggul untuk dataset tabular dengan class imbalance')
print('- Data preprocessing sudah selesai. Data siap untuk tahap modeling lanjutan (hyperparameter tuning, MLflow tracking)')


## 5.9 Persiapan Data Test

In [ ]:
# 9. Terapkan preprocessing yang sama pada test_df
def preprocess_test_data(test_df, train_columns):
    """
    Fungsi untuk melakukan preprocessing pada data test
    dengan transformasi yang sama seperti data train.
    """
    df_test = test_df.copy()
    
    # Drop kolom yang tidak diperlukan
    df_test = df_test.drop(columns=cols_to_drop, errors='ignore')
    
    # Binary encoding
    for col in binary_cols:
        if col in df_test.columns:
            df_test[col] = df_test[col].map({'Yes': 1, 'No': 0})
    
    # area_code ke string
    if 'area_code' in df_test.columns:
        df_test['area_code'] = df_test['area_code'].astype(str)
    
    # One-Hot Encoding
    df_test = pd.get_dummies(df_test, columns=multi_cat_cols, drop_first=False, dtype=int)
    
    # Tambahkan kolom yang ada di train tapi tidak di test (kasus OHE mismatch)
    for col in train_columns:
        if col not in df_test.columns:
            df_test[col] = 0
    
    # Hapus kolom yang ada di test tapi tidak di train
    df_test = df_test[[c for c in train_columns if c in df_test.columns]]
    
    # Feature scaling
    scale_cols_test = [c for c in scale_cols if c in df_test.columns]
    if scale_cols_test:
        df_test[scale_cols_test] = scaler.transform(df_test[scale_cols_test])
    
    return df_test

# Preprocessing data test
X_test_processed = preprocess_test_data(test_df, X_encoded.columns)
y_test = test_df['churn'] if 'churn' in test_df.columns else None

print('Data Test setelah preprocessing:')
print(f'X_test shape: {X_test_processed.shape}')
print(f'Kolom match dengan train: {X_test_processed.columns.tolist() == X_encoded.columns.tolist()}')

print('\n=== PREPROCESSING SELESAI ===')
print(f'Final X_train shape: {X_train.shape}')
print(f'Final X_val shape: {X_val.shape}')
print(f'Final X_test shape: {X_test_processed.shape}')
print('\nDataset siap untuk tahap Modeling selanjutnya.')


## 5.10 Menyimpan Data Preprocessing

Menyimpan data yang sudah dipreprocessing ke dalam folder `customer_churn_preprocessing/` agar dapat digunakan pada tahap modelling selanjutnya.

In [ ]:
# ============================================
# SAVE PREPROCESSED DATA
# ============================================
# Simpan data yang sudah dipreprocessing ke folder customer_churn_preprocessing/

import pickle

# Simpan train data
train_data = {
    'X_train': X_train,
    'X_val': X_val,
    'y_train': y_train,
    'y_val': y_val,
    'feature_names': X_train.columns.tolist(),
    'target_name': 'churn',
}

with open(f'{PREPROCESSING_DIR}/train_data.pkl', 'wb') as f:
    pickle.dump(train_data, f)
print(f'Train data saved: {PREPROCESSING_DIR}/train_data.pkl')

# Simpan test data
test_data = {
    'X_test': X_test_processed,
    'y_test': y_test,
}

with open(f'{PREPROCESSING_DIR}/test_data.pkl', 'wb') as f:
    pickle.dump(test_data, f)
print(f'Test data saved: {PREPROCESSING_DIR}/test_data.pkl')

# Simpan juga sebagai CSV untuk kemudahan inspeksi
X_train.to_csv(f'{PREPROCESSING_DIR}/X_train.csv', index=False)
X_val.to_csv(f'{PREPROCESSING_DIR}/X_val.csv', index=False)
y_train.to_csv(f'{PREPROCESSING_DIR}/y_train.csv', index=False)
y_val.to_csv(f'{PREPROCESSING_DIR}/y_val.csv', index=False)
X_test_processed.to_csv(f'{PREPROCESSING_DIR}/X_test.csv', index=False)
if y_test is not None:
    y_test.to_csv(f'{PREPROCESSING_DIR}/y_test.csv', index=False)
print(f'\nCSV files saved to: {PREPROCESSING_DIR}')

# Simpan scaler dan metadata untuk digunakan di modelling
preprocessing_metadata = {
    'scaler': scaler,
    'scale_cols': scale_cols,
    'cols_to_drop': cols_to_drop,
    'binary_cols': binary_cols,
    'multi_cat_cols': multi_cat_cols,
    'feature_names': X_train.columns.tolist(),
}

with open(f'{PREPROCESSING_DIR}/preprocessing_metadata.pkl', 'wb') as f:
    pickle.dump(preprocessing_metadata, f)
print(f'Preprocessing metadata saved: {PREPROCESSING_DIR}/preprocessing_metadata.pkl')

print('\n=== SEMUA DATA PREPROCESSING BERHASIL DISIMPAN ===')
print(f'Output directory: {PREPROCESSING_DIR}')
print('File yang dihasilkan:')
for f in os.listdir(PREPROCESSING_DIR):
    fpath = os.path.join(PREPROCESSING_DIR, f)
    size = os.path.getsize(fpath)
    print(f'  - {f} ({size:,} bytes)')
